# FANT 3 — 25M pretraining + SFT, NVIDIA-heavy mix (2026-04-23)

Small-scale run of fant3_20m preset (23.5M stored) on a single Colab A100 80 GB. No-truncation policy: both phases use `per_row` mode with `max_row_tokens = SEQ_LEN` so oversize rows are skipped rather than having their tails chopped off. Two phases:

* **Phase A — pretraining** (8000 steps, base LM + reasoning): FineWeb-Edu 35%, NVIDIA OpenMathReasoning 20%, NVIDIA OpenCodeReasoning-2 10%, NVIDIA OpenMathInstruct-2 10%, Opus-4.6 Reasoning 10%, Kimi K2.5 Distill 10%, Superior Reasoning S1 5%.
* **Phase B — post-training SFT** (4000 steps, chat + instruction following): NVIDIA Cascade-2 SFT math / chat / IF / science 60% total, NVIDIA Daring-Anteater 10%, Sonnet-4.6-120k 20%, NVIDIA OpenMathInstruct-2 10%.

Recipe (Tier D, calibrated for ~987M stored params):

| Knob | Value | Rationale |
|---|---|---|
| BATCH_SIZE | 2 | A100 ceiling with MoE+MoR+gc at T=1024 |
| SEQ_LEN | 1024 | Matches `cfg.max_seq_len` |
| GRAD_ACCUM | 4 | effective batch = 8 |
| PHASE_A_STEPS | 8000 | ~66M pretraining tokens |
| PHASE_B_STEPS | 4000 | ~33M SFT tokens |
| WARMUP_STEPS | 1800 | NeMo-style 15% of total |
| PEAK_LR | 1.2e-4 | below our NaN-at-3e-4 safety margin for 150m |
| GC | on | 2.6× VRAM reduction; required at 1B / T=1024 |
| OPTIM | bnb.optim.AdamW8bit | ~2 GB state vs 8 GB fp32 |

CERN-inspired opt-in flags (all landed 2026-04-21) active this run:
* `spinor_apollonian_enabled` — Kocik tangency spinor chirality
* `mor_lti_injection_enabled` + `mor_lti_apollonian_channel` — memory feeds MoR C channel
* `mor_adaptive_depth` — k_cap = min(max_depth, log₂ T+1)
* `mor_isrm_contractive` — Banach contractive refinement
* `use_gradient_checkpointing` — activations rematerialized

Expected wall time: ~10–12 h on A100 80 GB, ~36–46 GB peak VRAM.

## 0 · Setup: Drive mount, unzip, HF auth, deps

In [ ]:
# Cell 0.1 — Drive mount + locate fant_code.zip + unzip
import os, sys, shutil, zipfile, glob, time
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    zips = glob.glob('/content/drive/MyDrive/**/fant_code.zip', recursive=True)
    assert zips, 'Upload fant_code.zip to MyDrive first.'
    WORK_DIR = '/content/fant2'
    if os.path.isdir(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    os.makedirs(WORK_DIR, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(WORK_DIR)
    CKPT_DIR = '/content/drive/MyDrive/fant3_25m_nvidia'
    os.makedirs(CKPT_DIR, exist_ok=True)
    print('unzipped ->', WORK_DIR, '| ckpt dir:', CKPT_DIR)
else:
    WORK_DIR = os.getcwd()
    CKPT_DIR = os.path.join(WORK_DIR, 'output', 'fant3_25m_nvidia')
    os.makedirs(CKPT_DIR, exist_ok=True)
os.chdir(WORK_DIR); sys.path.insert(0, WORK_DIR)
print('cwd =', os.getcwd())

In [ ]:
# Cell 0.2 — PYTORCH_CUDA_ALLOC_CONF BEFORE torch import (Colab fragments badly without this)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# pip: datasets + bitsandbytes for 8-bit AdamW; scipy for eval utilities
import subprocess
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', *a])
try: import bitsandbytes  # noqa: F401
except Exception: _pip('bitsandbytes')
try: import datasets  # noqa: F401
except Exception: _pip('datasets>=2.19')
try: import scipy  # noqa: F401
except Exception: _pip('scipy')
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

In [ ]:
# Cell 0.3 — HF auth (only required for the gated Nemotron-CC-* datasets;
# the 11 datasets used in this notebook are all public, so this is optional).
import os
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login as hf_login
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF authed')
else:
    print('no HF_TOKEN set; public datasets will still work')

In [ ]:
# Cell 0.4 — import the repo
from fant3.config import fant3_20m as fant3_25m, FANT3Config   # 20m preset = 23.5M, called 25m loosely
from fant3.model.fant3_model import FANT3Model
from fant3.training import schedule_multiplier, precondition_router_grads_
from fant2.data.formats import DatasetFormat
from fant2.data.streaming import InterleavedMultiDatasetStream, HuggingFaceStream
print('imports OK')

## 1 · Build the 1B config with CERN-inspired flags on

Defaults give ~987M stored params. The opt-in flags turn on the physics-derived behaviors landed 2026-04-21.

In [ ]:
cfg = fant3_25m()
cfg.max_seq_len = 16384  # 2026-04-23: raised to 16384 so ALL rows fit (max observed 14136); no truncation, no skipping
# CERN-inspired opt-ins (all default False in config.py)
cfg.use_gradient_checkpointing  = True   # required at 1B + T=1024
cfg.mor_lti_injection_enabled   = True
cfg.mor_spectral_constraint     = True   # keeps rho(A)<1
cfg.mor_loop_index_enabled      = True
cfg.mor_lti_apollonian_channel  = True   # feeds Apollonian memory into MoR C channel
cfg.mor_adaptive_depth          = True
cfg.mor_isrm_contractive        = True   # Banach contractive refinement
# spinor_apollonian_enabled already True by default in fant3_1b()

# Tier 1/2 CE stabilisation (landed 2026-04-22) - see investigation memo
cfg.lm_head_logit_cap               = 30.0   # Gemma-2 style bf16 overflow guard
cfg.apollonian_channel_warmup_steps = 500    # delay MoR C-channel until memory embeddings meaningful

print(cfg)

In [ ]:
# Cell 1.2 — build + param audit (before optimiser allocation)
torch.manual_seed(0); np.random.seed(0)
model = FANT3Model(cfg).to(dtype=DTYPE, device=DEVICE)
n_stored = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'stored params    : {n_stored/1e6:.2f} M')
print(f'trainable params : {n_trainable/1e6:.2f} M')
assert 15e6 < n_stored < 50e6, f'expected ~25M (fant3_20m preset = 23.5M), got {n_stored}'

# Tier 3 (2026-04-22): promote tied tok_emb/lm_head to fp32 to escape the bf16
# update-quantisation ceiling. The tied embedding is the gradient bottleneck:
# |g_i| ~ 1/(B*T) ~ 1.2e-4 with ||g|| ~ 440 gives concentration r_i ~ 3e-7,
# and cumulative drift over 12000 steps with ANY clip stays below the bf16
# relative threshold eps = 2^-7 ~ 7.8e-3. In fp32 the threshold is 2^-23 ~ 1.2e-7,
# making the inequality satisfiable at any reasonable clip. Extra VRAM: ~67 MB
# (vocab * dim * 2 extra bytes for fp32 vs bf16).
TIER3_FP32_TIED_EMB = True   # set False to revert
if TIER3_FP32_TIED_EMB:
    # tok_emb.weight IS lm_head.weight (tied), so this upgrades both paths.
    with torch.no_grad():
        model.tok_emb.weight.data = model.tok_emb.weight.data.float()
    assert model.lm_head.weight.data_ptr() == model.tok_emb.weight.data_ptr(), 'tying broke'
    n_stored_after = sum(p.numel() for p in model.parameters())
    extra_mb = (n_stored_after * 4 - n_stored * 2) / 1e6 if False else cfg.vocab_size * cfg.dim * 2 / 1e6
    print(f'tied emb/lm_head upgraded: dtype={model.tok_emb.weight.dtype}  extra VRAM ~{extra_mb:.0f} MB')

## 2 · Tokenizer + decontamination filter

Uses the BPE tokenizer trained on the 6-source mix and the 13-gram SHA-1 hash cache (457 k hashes) so training never sees a GSM8K / MMLU / MATH-500 test-set n-gram.

In [ ]:
from tokenizers import Tokenizer
TOK_PATH = os.path.join(WORK_DIR, 'output', 'tokenizer', 'tokenizer_v2.json')
tok = Tokenizer.from_file(TOK_PATH)
VOCAB_SIZE = tok.get_vocab_size()
# Explicit None checks — token_to_id returns 0 for <|pad|> which is truthy in reality
# but falsy to Python's `or` operator, masking any real miss.
PAD_ID = tok.token_to_id('<|pad|>')
EOS_ID = tok.token_to_id('<|eos|>')
if PAD_ID is None: PAD_ID = 0
if EOS_ID is None: EOS_ID = 1
print(f'vocab={VOCAB_SIZE}  pad_id={PAD_ID}  eos_id={EOS_ID}')

# Tokenizer is the source of truth. Align cfg.vocab_size to it and rebuild the
# model if it was already constructed (cell 1.2 sets a default vocab of 32768;
# tokenizer_v2.json trained to 32736, the 32-token gap is reserved unused slots
# in the BPE build — see reference_tokenizer_v2 memory).
if VOCAB_SIZE != cfg.vocab_size:
    print(f'aligning cfg.vocab_size {cfg.vocab_size} -> {VOCAB_SIZE} (tokenizer is authoritative)')
    cfg.vocab_size = VOCAB_SIZE
    if 'model' in globals():
        torch.manual_seed(0); np.random.seed(0)
        model = FANT3Model(cfg).to(dtype=DTYPE, device=DEVICE)
        n_stored = sum(p.numel() for p in model.parameters())
        print(f'  rebuilt model with aligned vocab — stored = {n_stored/1e6:.2f} M')

In [ ]:
# Cell 2.2 — decontamination filter (uses scripts/decontaminate.py global cache)
from scripts.decontaminate import is_contaminated, build_hash_cache
_cache = build_hash_cache(rebuild=False)
_n_hashes = sum(len(v) for v in _cache.values())
print(f'decontamination cache loaded: {_n_hashes} n-gram hashes across {len(_cache)} benchmarks')

## 3 · Phase A stream — pretraining (base LM + reasoning)

Weights sum to 1.00. FineWeb-Edu provides the long-tail web text; NVIDIA OpenMathReasoning / OpenCodeReasoning / OpenMathInstruct give HEP-trained reasoning traces; Kimi + Opus + Superior add Claude-family teacher traces.

In [ ]:
# CURRICULUM_PATCH_V1
# Curriculum preset — switch between mixes without editing dataset lists.
# Available presets (see fant3/training/curriculum.py):
#   'legacy_2phase'      — the original 2-phase mix (bit-identical default)
#   'deepinsight_3phase' — Apprentice/Journeyman/Expert, arxiv:2604.16278
#   'flat_1phase'        — no curriculum, single mix throughout (A/B control)
CURRICULUM_NAME = 'legacy_2phase'

from fant3.training import build_curriculum
CURRICULUM = build_curriculum(CURRICULUM_NAME)
print(f'Curriculum: {CURRICULUM.name} with {len(CURRICULUM.phases)} phase(s)')
for i, _p in enumerate(CURRICULUM.phases):
    print(f'  phase {i} {_p.name!r}: end_frac={_p.end_frac:.3f} datasets={list(_p.datasets)}')

# Back-compat aliases — cells below still read PHASE_A_*/PHASE_B_*.
# For non-2-phase curricula, PHASE_B_* mirrors the LAST phase and the
# notebook's 2-sampler architecture only sees phase 0 (A) and last (B);
# intermediate phases are ignored unless you use scripts/runpod_train.py.
PHASE_A_DATASETS = list(CURRICULUM.phases[0].datasets)
PHASE_A_WEIGHTS  = list(CURRICULUM.phases[0].weights)
PHASE_B_DATASETS = list(CURRICULUM.phases[-1].datasets)
PHASE_B_WEIGHTS  = list(CURRICULUM.phases[-1].weights)


## 4 · Phase B stream — post-training SFT (chat + instruction)

NVIDIA Cascade-2 SFT subsets (math / chat / instruction_following / science) + Daring-Anteater teach the chat/assistant persona. Sonnet-4.6-120k + OpenMathInstruct-2 keep quality-graded distillation traces in the mix so the model doesn't forget reasoning while acquiring chat.

In [ ]:
# Phase B is derived from CURRICULUM.phases[-1] (see patch in cell above).
# Kept as a separate cell so per-phase audit/plot cells still execute.
print('PHASE_B_DATASETS:', PHASE_B_DATASETS)
print('PHASE_B_WEIGHTS: ', PHASE_B_WEIGHTS)


## 4.5 · Measure per-phase token-length distribution

Samples 50 docs from each phase's stream. Run this before the recipe-knobs cell so `SEQ_LEN_A` / `SEQ_LEN_B` come from real data, not a hardcoded guess.

In [ ]:
# Cell 4.5 - Measure per-dataset token-length distribution
# Fresh HuggingFaceStream per registry entry so (a) one failing dataset doesn't
# crash the phase, and (b) a low-weight dataset still gets enough samples to
# report a real distribution. Reports per-dataset P50/P75/P95/P99/max, then
# a weighted phase P95 computed only from the datasets that succeeded.
from fant2.data.streaming import HuggingFaceStream
from fant2.data.registry import ALL_DATASETS
import numpy as np

def _per_dataset_lengths(ds_key, n=20):
    """Sample n docs from one registered dataset. Returns [] on any failure."""
    if ds_key not in ALL_DATASETS:
        print(f'    {ds_key:32s} NOT IN REGISTRY (skipping)')
        return []
    e = ALL_DATASETS[ds_key]
    try:
        s = HuggingFaceStream(
            dataset_name=e.hf_id, dataset_config=e.config, split=e.split,
            text_key=e.text_key, format_type=e.format,
            input_key=e.input_key, output_key=e.output_key,
        )
        lens = []
        for i, text in enumerate(s):
            if i >= n: break
            if not text: continue
            lens.append(len(tok.encode(text).ids))
        return lens
    except Exception as ex:
        print(f'    {ds_key:32s} LOAD FAILED ({type(ex).__name__}: {str(ex)[:80]})')
        return []

def _measure_phase(names, weights, tag, per_ds_n=20, cap=None):
    print(f'=== {tag} ===')
    successful = []     # (weight, p50, p95, lengths) for each successful dataset
    truncation_warnings = []
    for name, w in zip(names, weights):
        lens = _per_dataset_lengths(name, n=per_ds_n)
        if not lens:
            continue
        a = np.array(lens)
        p50, p75, p95, p99, mx = (int(np.percentile(a, q)) for q in (50, 75, 95, 99, 100))
        flag = ''
        if cap is not None:
            if p50 > cap * 2:
                flag = '  !! P50 > 2x cap: majority of rows will be severely truncated'
                truncation_warnings.append((name, w, p50, p95, cap, 'severe'))
            elif p50 > cap:
                flag = '  !  P50 > cap: at least half of rows will be truncated'
                truncation_warnings.append((name, w, p50, p95, cap, 'partial'))
        print(f'    {name:32s} w={w:.2f} n={len(a):3d}  P50={p50:5d}  P75={p75:5d}  P95={p95:5d}  P99={p99:5d}  max={mx:5d}{flag}')
        successful.append((w, p50, p95, lens))
    if not successful:
        print(f'    {tag}: no successful samples'); return None, None, truncation_warnings
    flat_pool = np.concatenate([np.array(l) for (_, _, _, l) in successful])
    flat_p95 = int(np.percentile(flat_pool, 95))
    wsum = sum(w for (w, _, _, _) in successful)
    weighted_p95 = sum(w * p95 for (w, _, p95, _) in successful) / wsum
    print(f'    {tag}: flat_P95={flat_p95}  weighted_P95={weighted_p95:.0f}  n_total={len(flat_pool)}  weight_covered={wsum:.2f}')
    return flat_p95, weighted_p95, truncation_warnings

cap = cfg.max_seq_len
flat_A, weighted_A, warn_A = _measure_phase(PHASE_A_DATASETS, PHASE_A_WEIGHTS, tag='Phase A', per_ds_n=20, cap=cap)
flat_B, weighted_B, warn_B = _measure_phase(PHASE_B_DATASETS, PHASE_B_WEIGHTS, tag='Phase B', per_ds_n=20, cap=cap)

def _suggest(flat_p95, weighted_p95, cap, round_to=128):
    if flat_p95 is None:
        return cap
    raw = max(int(flat_p95), int(weighted_p95 or 0))
    return min(cap, max(256, ((raw + round_to - 1) // round_to) * round_to))

SEQ_LEN_A_SUGGEST = _suggest(flat_A, weighted_A, cap=cap)
SEQ_LEN_B_SUGGEST = _suggest(flat_B, weighted_B, cap=cap)
print(f"\nsuggested SEQ_LEN_A = {SEQ_LEN_A_SUGGEST}   (concat-pack; pretraining)")
print(f"suggested SEQ_LEN_B = {SEQ_LEN_B_SUGGEST}   (per-row-pad; SFT)")

# Prominent warning if any SFT dataset will lose >=half its rows. For SFT this
# is especially bad because answer tokens usually live at the END of a trace,
# so truncation destroys the training signal.
all_warnings = (warn_A or []) + (warn_B or [])
if all_warnings:
    print('\n' + '='*72)
    print('TRUNCATION WARNING - the following datasets will lose most of their')
    print(f'content at SEQ_LEN cap = {cap}:')
    for (name, w, p50, p95, c, sev) in all_warnings:
        pct = int(100 * c / max(p50, 1))
        print(f'  [{sev:>7}] {name:32s} w={w:.2f}  keeps ~{pct}% of P50 row ({c}/{p50} tokens)')
    print('Options:')
    print('  1. RAISE cfg.max_seq_len (requires rebuilding model + more VRAM at same B)')
    print('  2. REDUCE weight of the long-tail dataset below (or drop entirely)')
    print('  3. ACCEPT and set MAX_ROW_TOKENS below so long rows are skipped,')
    print('     not truncated - better signal, fewer samples per epoch')
    print('='*72)

print("\nOverride in the next cell (SEQ_LEN_A = ...) if you want something different.")

## 5 · Batch sampler (decontamination-wrapped)

In [ ]:
def make_batch_sampler(stream, batch_size, seq_len, pad_id, eos_id,
                       contamination_filter=True, pack_mode='concat',
                       max_row_tokens=None):
    """Yields (ids, targets) tensors of shape (B, seq_len).

    pack_mode='concat'  (Phase A, pretraining):
        Concatenate documents with <|eos|> separators until the buffer reaches
        seq_len. Standard LM pretraining pattern.

    pack_mode='per_row' (Phase B, SFT):
        One document per sample. Behavior for oversized docs is controlled by
        `max_row_tokens`:
            * None (default) - truncate to seq_len-1 + <|eos|>. Simple but
              loses answer tokens when the real doc is longer than seq_len.
            * int  - skip any doc whose encoded length is > max_row_tokens.
              Preserves training signal quality at the cost of discarding
              some rows. Useful for SFT with long-tail sources (Cascade-2
              math, chat) where truncation would drop the answer.

    Both modes share the decontamination filter.
    """
    assert pack_mode in ('concat', 'per_row')
    it = iter(stream)
    def _next_clean_text():
        while True:
            text = next(it)
            if not text: continue
            if contamination_filter and is_contaminated(text):
                continue
            return text
    while True:
        batch_ids = torch.full((batch_size, seq_len), pad_id, dtype=torch.long)
        for b in range(batch_size):
            tokens = []
            if pack_mode == 'concat':
                while len(tokens) < seq_len:
                    text = _next_clean_text()
                    ids = tok.encode(text).ids
                    if not ids: continue
                    tokens.extend(ids)
                    tokens.append(eos_id)
                row = tokens[:seq_len]
            else:  # per_row
                while not tokens:
                    text = _next_clean_text()
                    ids = tok.encode(text).ids
                    if not ids:
                        continue
                    if max_row_tokens is not None and len(ids) > max_row_tokens:
                        # Skip docs that would be severely truncated; preserves SFT signal.
                        continue
                    tokens = ids[:seq_len - 1] + [eos_id]
                row = tokens + [pad_id] * (seq_len - len(tokens))
            batch_ids[b] = torch.tensor(row, dtype=torch.long)
        targets = batch_ids.clone()
        targets[targets == pad_id] = -100  # CE ignores pads
        yield batch_ids, targets

## 6 · Training loop (unified, phase-aware)

Single loop that consumes `stream_A` for the pretraining phase, then switches to `stream_B` for SFT. LR schedule spans both phases with a single long warmup + Litim/cosine decay over the full `TOTAL_STEPS`.

In [ ]:
# Recipe knobs - Tier D for 1B, calibrated for A100 80 GB
# 2026-04-22: BATCH_SIZE reduced 2 -> 1 (MoE expert W_up_sel materialization
# peaked ~38 GB at B=2 worst-case routing skew, OOM'd mid-training). Compensated
# with GRAD_ACCUM_STEPS 4 -> 8 so effective batch stays at 8.
# Tier 1/2 CE stabilisation: GRAD_CLIP tightened 1.0 -> 0.5, WARMUP 1800 -> 2400,
# added z-loss wiring + Fisher-preconditioner warmup + grad-norm monitoring.
BATCH_SIZE        = 1           # 2026-04-23 (T=16384 update): B=1 to keep MoE materialization peaks under control; H100 80GB fits comfortably
GRAD_ACCUM_STEPS  = 4           # effective batch = 4 (T=16384 * 4 = 65k tokens/step, well-tuned for 25M)
SEQ_LEN_A         = SEQ_LEN_A_SUGGEST    # pretrain, concat-packing
SEQ_LEN_B         = SEQ_LEN_B_SUGGEST    # SFT, one-row-per-sample
# 2026-04-23 T=16384 budget: 65k tokens/step * 5000 steps = 325M training tokens
# (13 tokens/param for 25M, close to Chinchilla ~20). Phase A:B split 60:40.
PHASE_A_STEPS     = 3000
PHASE_B_STEPS     = 2000
TOTAL_STEPS       = PHASE_A_STEPS + PHASE_B_STEPS
WARMUP_STEPS      = 250         # 2026-04-23 (T=16384): scaled down from 500 proportional to reduced TOTAL_STEPS
PEAK_LR           = 2.0e-4       # 2026-04-22: bumped 1.2e-4->2e-4 with fp32 tied emb headroom; matches NeMo recipe; if NaN, back off to 1.5e-4
GRAD_CLIP         = 10.0        # 2026-04-22: raised 0.5 -> 10.0 after diagnosing clip-strangulation. Architecture grad_norm baseline is ~400-500 (MoR x2 + tied emb + MoE + Cerebellum + AHN summing), clip=0.5 was retaining 0.1% signal -> no learning visible at step 375. clip=10 retains ~2%, matches what 742m Tier C had proportional to its grad_norm=~150 + clip=1.0.
SCHEDULE_SHAPE    = 'litim'
LOG_EVERY         = 25
CKPT_EVERY        = 500
STORE_EVERY       = 50
FISHER_PRECOND    = False    # 2026-04-22: rescales router grads to mag ~10/element, grad_norm explodes to 500+ and clip=0.5 crushes effective update 1000x -> no learning. Turn back on only after calibrating a separate LR group for router params.
FISHER_WARMUP_STEPS = 100       # skip precondition for first N steps (EMA warmup)
Z_LOSS_COEF       = 1e-4        # OLMoE-style router z-loss weight; kills router drift
GRAD_NORM_ALERT_X = 3.0         # log warning when grad_norm > X * running median
CE_PROBE_EVERY    = 0           # 2026-04-22: disabled - probe was OOMing at B=4 due to Matryoshka MoE materialization (77 GB per block). Re-enable after fixing probe batch to B=1.
print(f'total={TOTAL_STEPS}  warmup={WARMUP_STEPS}  peak_lr={PEAK_LR:.1e}  schedule={SCHEDULE_SHAPE}')
print(f'seq_len: Phase A = {SEQ_LEN_A} (concat)   Phase B = {SEQ_LEN_B} (per_row)')
print(f'batch: B={BATCH_SIZE} accum={GRAD_ACCUM_STEPS} effective={BATCH_SIZE*GRAD_ACCUM_STEPS}')
print(f'stability: clip={GRAD_CLIP}  z_loss={Z_LOSS_COEF}  fisher_warmup={FISHER_WARMUP_STEPS}  ce_probe_every={CE_PROBE_EVERY}')

## 6.1b · Per-domain CE probe (held-out batches)

Samples a fixed held-out batch from each phase dataset so the training loop can measure CE per source on identical samples over time. This is what tells you whether CE is actually moving toward 1.x on clean prose (FineWeb) versus getting stuck on reasoning/chat traces where CE is intrinsically higher.

In [ ]:
# Cell 6.1b - build per-domain CE probe (held-out samples, not in training path)
from fant2.data.streaming import HuggingFaceStream
from fant2.data.registry import ALL_DATASETS

def _build_probe(ds_key, batch_size=4, seq_len=SEQ_LEN_A, pad_id=PAD_ID, eos_id=EOS_ID):
    """Grab `batch_size` rows from one dataset, tokenize, pad/trunc to seq_len,
    return (ids, targets) on DEVICE. Returns None if dataset unavailable."""
    if ds_key not in ALL_DATASETS:
        return None
    e = ALL_DATASETS[ds_key]
    try:
        s = HuggingFaceStream(dataset_name=e.hf_id, dataset_config=e.config,
                              split=e.split, text_key=e.text_key, format_type=e.format,
                              input_key=e.input_key, output_key=e.output_key)
    except Exception:
        return None
    rows = []
    it = iter(s)
    for i in range(batch_size * 3):  # overhead for skip of empty/contaminated
        try: text = next(it)
        except StopIteration: break
        if not text: continue
        ids = tok.encode(text).ids
        if not ids: continue
        ids = ids[:seq_len - 1] + [eos_id]
        ids = ids + [pad_id] * (seq_len - len(ids))
        rows.append(ids)
        if len(rows) >= batch_size: break
    if len(rows) < batch_size:
        return None
    batch = torch.tensor(rows, dtype=torch.long, device=DEVICE)
    tgt = batch.clone()
    tgt[tgt == pad_id] = -100
    return (batch, tgt)

CE_PROBES = {}
for ds_key in set(PHASE_A_DATASETS) | set(PHASE_B_DATASETS):
    probe = _build_probe(ds_key, batch_size=1, seq_len=min(SEQ_LEN_A, 1024))
    if probe is not None:
        CE_PROBES[ds_key] = probe
print(f'CE probe built for {len(CE_PROBES)} domains: {sorted(CE_PROBES.keys())}')

@torch.no_grad()
def ce_probe_eval(model):
    """Return {ds_key: ce_float} for every probe. Sets model.eval() temporarily.
    2026-04-22: added per-probe empty_cache to avoid the 30 GB post-probe VRAM
    spike, and wrap each probe in its own no_grad + detach to keep graph refs
    from leaking into the next training step."""
    was_training = model.training
    model.eval()
    out = {}
    for k, (ids, tgt) in CE_PROBES.items():
        try:
            r = model(ids, targets=tgt)
            l = float(r['loss'].detach().float())
            if l == l:  # not NaN
                out[k] = l
            del r
        except Exception as e:
            out[k] = float('nan')
            print(f'  [probe-err] {k}: {type(e).__name__}: {str(e)[:80]}')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if was_training: model.train()
    return out

In [ ]:
# Cell 6.2 — optimiser + scheduler
import bitsandbytes as bnb
optim = bnb.optim.AdamW8bit(model.parameters(), lr=PEAK_LR,
                            betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8)
def lr_at(step):
    mul = schedule_multiplier(step, WARMUP_STEPS, TOTAL_STEPS, shape=SCHEDULE_SHAPE)
    return PEAK_LR * mul
print('optimiser:', type(optim).__name__, '| LR at step 0:', lr_at(0),
      '| at warmup_end:', lr_at(WARMUP_STEPS), '| at total:', lr_at(TOTAL_STEPS))

In [ ]:
# Cell 6.3 - training loop (Tier 1/2 CE stabilisation + two-tier ckpt + resume + NaN guard)
import gc as _gc
import glob as _glob
import math as _math

# MAX_ROW_TOKENS in per_row mode: set this to SEQ_LEN_B to drop docs longer
# than the seq_len (avoids "problem setup + no answer" truncation in SFT).
MAX_ROW_TOKENS_B = SEQ_LEN_B   # change to None if you prefer truncate-over-skip
# 2026-04-23: Phase A switched to per_row mode with skip-long (no truncation).
# Side effect: shorter docs get padded, longer docs skipped; 25M model fast enough to absorb the waste.
MAX_ROW_TOKENS_A = SEQ_LEN_A
sampler_A = make_batch_sampler(stream_A, BATCH_SIZE, SEQ_LEN_A, PAD_ID, EOS_ID, pack_mode='per_row', max_row_tokens=MAX_ROW_TOKENS_A)
sampler_B = make_batch_sampler(stream_B, BATCH_SIZE, SEQ_LEN_B, PAD_ID, EOS_ID, pack_mode='per_row', max_row_tokens=MAX_ROW_TOKENS_B)
fisher_state = {}
loss_hist  = []
grad_norm_hist = []   # for running-median alert

# -------------------- checkpoint destinations --------------------
LOCAL_CKPT_DIR  = '/content/ckpts_local' if IN_COLAB else os.path.join(CKPT_DIR, '_local')
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
ROLLING_KEEP    = 3
DRIVE_MILESTONE_EVERY = 2000
RESUME_FROM     = None

# -------------------- NaN guard --------------------
MAX_CONSECUTIVE_NAN_STEPS = 3
NAN_STEPS_TOTAL           = 0
CONSECUTIVE_NAN           = 0

def _save(path, model, optim, step, include_optim=True, extra=None):
    payload = {'model': model.state_dict(), 'step': step, 'cfg': cfg.__dict__,
               'extra': extra or {}}
    if include_optim:
        payload['optim'] = optim.state_dict()
    torch.save(payload, path)
    sz = os.path.getsize(path) / 1e9
    print(f'  [ckpt] step={step} -> {path}  size={sz:.2f} GB  {"(+optim)" if include_optim else "(weights only)"}')

def _rolling_trim(local_dir, keep=ROLLING_KEEP):
    files = sorted(_glob.glob(os.path.join(local_dir, 'step_*.pt')), key=os.path.getmtime)
    for old in files[:-keep]:
        try:
            os.remove(old); print(f'  [rolling] removed {os.path.basename(old)}')
        except OSError: pass

# -------------------- optional resume --------------------
start_step = 0
if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
    print(f'resuming from {RESUME_FROM}')
    state = torch.load(RESUME_FROM, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state['model'])
    if 'optim' in state: optim.load_state_dict(state['optim'])
    start_step = int(state.get('step', 0))
    loss_hist = list(state.get('extra', {}).get('loss_hist', []))
    print(f'  loaded model + optim, resume at step {start_step+1}')

# -------------------- the loop --------------------
start = time.time()
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()

for step in range(start_step + 1, TOTAL_STEPS + 1):
    # Propagate step to submodules (MoR Apollonian-channel warmup gate)
    model.set_global_step(step)

    sampler = sampler_A if step <= PHASE_A_STEPS else sampler_B
    phase_tag = 'A' if step <= PHASE_A_STEPS else 'B'
    cur_lr = lr_at(step)
    for g in optim.param_groups: g['lr'] = cur_lr

    model.train(); step_loss = 0.0; step_z_loss = 0.0
    optim.zero_grad(set_to_none=True)
    out = None; router_info_snapshot = None
    n_nan_micros = 0; n_ok_micros = 0
    max_logit_abs = 0.0; max_router_logit_abs = 0.0

    for micro in range(GRAD_ACCUM_STEPS):
        ids, targets = next(sampler)
        ids, targets = ids.to(DEVICE), targets.to(DEVICE)
        store_now = (step % STORE_EVERY == 0) and (micro == 0)
        out = model(ids, targets=targets, store_to_memory=store_now)

        # Sum router z-loss from every suffix MoE block + the MoR shared block
        z_sum = 0.0
        for ri in (out.get('router_infos') or []):
            z = ri.get('z_loss')
            if z is not None: z_sum = z_sum + z
            mp_lg = ri.get('mp_logits')
            if mp_lg is not None:
                mla = float(mp_lg.abs().max())
                if mla > max_router_logit_abs: max_router_logit_abs = mla

        total_loss = out['loss'] + Z_LOSS_COEF * z_sum
        loss_scaled = total_loss / GRAD_ACCUM_STEPS

        if torch.isfinite(loss_scaled):
            loss_scaled.backward()
            step_loss   += float(out['loss'])
            step_z_loss += float(z_sum) if isinstance(z_sum, torch.Tensor) else 0.0
            n_ok_micros += 1
            # Track max |logit| post-soft-cap if enabled
            if 'logits' in out:
                mla = float(out['logits'].abs().max())
                if mla > max_logit_abs: max_logit_abs = mla
        else:
            n_nan_micros += 1

        ri = out.get('router_infos') or []
        if ri and 'mp_replicon' in ri[0]:
            router_info_snapshot = float(ri[0]['mp_replicon'])
        del ids, targets, total_loss, loss_scaled

    # --------- step policy: NaN guard + Fisher + clip + optim.step ---------
    if n_ok_micros == 0:
        optim.zero_grad(set_to_none=True)
        loss_hist.append(float('nan'))
        NAN_STEPS_TOTAL += 1; CONSECUTIVE_NAN += 1
        print(f'  [NaN] step={step} all {GRAD_ACCUM_STEPS} micros NaN; skipped optim.step ({CONSECUTIVE_NAN}/{MAX_CONSECUTIVE_NAN_STEPS} consecutive)')
    else:
        if n_nan_micros > 0:
            print(f'  [NaN-mix] step={step} {n_nan_micros}/{GRAD_ACCUM_STEPS} micros NaN; applying partial grads')
            NAN_STEPS_TOTAL += 1
        # Fisher precondition only after warmup (EMA needs ~100 steps to stabilise)
        if FISHER_PRECOND and step >= FISHER_WARMUP_STEPS:
            precondition_router_grads_(model, fisher_state)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        grad_norm_hist.append(float(gn))
        optim.step()
        loss_hist.append(step_loss / max(n_ok_micros, 1))
        CONSECUTIVE_NAN = 0

        # Running-median alert: warn if grad_norm suddenly spikes
        if len(grad_norm_hist) >= 50:
            import statistics as _stats
            med = _stats.median(grad_norm_hist[-50:])
            if gn > GRAD_NORM_ALERT_X * med and med > 1e-6:
                print(f'  [grad-spike] step={step} grad_norm={gn:.3f} = {gn/med:.1f}x running-median (median_50={med:.3f})')

    if CONSECUTIVE_NAN >= MAX_CONSECUTIVE_NAN_STEPS:
        print(f'STOP: {CONSECUTIVE_NAN} consecutive all-NaN steps at step {step}. Lower PEAK_LR or check for bad data; resume via RESUME_FROM.')
        break

    # Aggressive cache flush every 4 steps to fight 1B-scale fragmentation
    if DEVICE == 'cuda' and step % 4 == 0:
        torch.cuda.empty_cache()

    # ------------------- logging ---------------------
    if step % LOG_EVERY == 0 or step in (start_step + 1, PHASE_A_STEPS + 1):
        elapsed = time.time() - start
        vram = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
        mstats = model.memory.get_stats() if hasattr(model, 'memory') else {}
        cur_sl = SEQ_LEN_A if phase_tag == 'A' else SEQ_LEN_B
        cur_loss = loss_hist[-1]
        loss_display = cur_loss if not (isinstance(cur_loss, float) and _math.isnan(cur_loss)) else float('nan')
        cur_gn = grad_norm_hist[-1] if grad_norm_hist else 0.0
        print(f'[{phase_tag} T={cur_sl}] step={step:5d} lr={cur_lr:.2e} loss={loss_display:.4f} z={step_z_loss:.3f} '
              f'gn={cur_gn:.2f} max|logit|={max_logit_abs:.1f} max|rtr|={max_router_logit_abs:.1f} '
              f'vram={vram:.1f}GB chirality={mstats.get("chirality_balance", 0.0):.3f} '
              f'replicon={router_info_snapshot if router_info_snapshot is not None else 0.0:+.2f} '
              f'nan_total={NAN_STEPS_TOTAL} elapsed={elapsed/60:.1f}m')
        if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()

    # ------------------- per-domain CE probe ---------------------
    if CE_PROBE_EVERY > 0 and step % CE_PROBE_EVERY == 0 and len(CE_PROBES) > 0:
        probe_ce = ce_probe_eval(model)
        # print as one compact line, sorted by CE ascending (best first)
        items = sorted(probe_ce.items(), key=lambda kv: kv[1] if kv[1] == kv[1] else 99.0)
        probe_line = '  [probe] ' + '  '.join(f'{k[:16]}={v:.2f}' for k, v in items[:6])
        print(probe_line)

    # -------------------- two-tier ckpt --------------------
    is_milestone = (step == PHASE_A_STEPS) or (step == TOTAL_STEPS) or (step % DRIVE_MILESTONE_EVERY == 0)
    is_rolling   = (step % CKPT_EVERY == 0)
    if is_rolling:
        _save(os.path.join(LOCAL_CKPT_DIR, f'step_{step:05d}.pt'), model, optim, step, include_optim=True,
              extra={'loss_hist': loss_hist[-CKPT_EVERY:], 'phase': phase_tag, 'nan_steps_total': NAN_STEPS_TOTAL,
                     'grad_norm_hist': grad_norm_hist[-CKPT_EVERY:]})
        _rolling_trim(LOCAL_CKPT_DIR, keep=ROLLING_KEEP)
    if is_milestone:
        _save(os.path.join(CKPT_DIR, f'step_{step:05d}.pt'), model, optim, step, include_optim=True,
              extra={'loss_hist': loss_hist[-DRIVE_MILESTONE_EVERY:], 'phase': phase_tag,
                     'milestone': 'phase_A_end' if step == PHASE_A_STEPS else ('final' if step == TOTAL_STEPS else 'periodic'),
                     'nan_steps_total': NAN_STEPS_TOTAL})
        _gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    del out; router_info_snapshot = None

print(f'training complete in {(time.time()-start)/3600:.2f} h')
print(f'NaN steps: {NAN_STEPS_TOTAL} / {TOTAL_STEPS}')
print(f'local ckpts:  {LOCAL_CKPT_DIR}  (rolling, last {ROLLING_KEEP})')
print(f'drive ckpts:  {CKPT_DIR}         (milestones only)')

## 7 · Final save + plot

In [ ]:
# Cell 7 - final ckpt under a stable name + loss plot
# _save is defined in cell 6.3; this cell runs AFTER 6.3 so it's in scope.
final_path = os.path.join(CKPT_DIR, 'final.pt')
_save(final_path, model, optim, TOTAL_STEPS, include_optim=True,
      extra={'phase_A_steps': PHASE_A_STEPS, 'phase_B_steps': PHASE_B_STEPS,
             'nan_steps_total': NAN_STEPS_TOTAL})
print('FINAL -> ', final_path)

# Loss plot with NaN-aware masking (NaN rows drawn as gaps)
try:
    import matplotlib.pyplot as plt
    import numpy as _np
    arr = _np.array([(x if x == x else _np.nan) for x in loss_hist], dtype=float)  # NaN preserved
    plt.figure(figsize=(9, 3.5))
    plt.plot(arr, lw=0.8, alpha=0.75)
    plt.axvline(PHASE_A_STEPS, ls='--', c='grey', label='phase A -> B handoff')
    plt.xlabel('step'); plt.ylabel('CE loss')
    plt.title(f'FANT 3 1B - NVIDIA-heavy pretrain + SFT (NaN steps = {NAN_STEPS_TOTAL})')
    plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print('plot skipped:', e)

## 8 · Post-training benchmarks (optional)

Runs the unified eval harness on GSM8K 200 + MMLU 500 + MATH-500 with Wilson 95% CIs. Skip if you want a cheap loss-curve-only run.

In [ ]:
import subprocess
cmd = [sys.executable, 'scripts/eval_benchmarks.py',
       '--ckpt', final_path,
       '--scale', '1b',
       '--gsm8k-n', '200', '--mmlu-n', '500', '--math500-n', '100']
print(' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout); print(res.stderr)

## 9 · Quality probe (free-form generation)

In [ ]:
PROMPTS = [
    'Question: Alice has 3 apples, Bob gives her 7 more and takes back 2. How many does she have?\nAnswer:',
    'Write a short haiku about attention mechanisms:',
    'def fibonacci(n):\n    ',
    'Explain briefly why the Descartes circle theorem matters for Apollonian packings:',
]
model.eval()
for p in PROMPTS:
    ids = torch.tensor(tok.encode(p).ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    out_ids = ids.clone()
    with torch.no_grad():
        for _ in range(80):
            logits = model(out_ids)['logits'][:, -1]
            nxt = torch.multinomial(torch.softmax(logits / 0.8, dim=-1), 1)
            out_ids = torch.cat([out_ids, nxt], dim=-1)
            if int(nxt.item()) in (EOS_ID, PAD_ID): break
    gen = tok.decode(out_ids[0].tolist()[len(ids[0]):])
    print('>', p)
    print(' ', gen.replace('\n', ' ')[:240])
    print()

---
## Summary

If the run gets this far with `loss_hist[-1] < 5.0`, the 1B target + NVIDIA-heavy pretraining + SFT + CERN-flags opt-in are all working. Next steps after this run:

1. Push `final.pt` to a separate Drive folder (don't overwrite the 742m Tier C checkpoint at 770M).
2. Run `scripts/fss_collapse.py --ladder fant_ladder.json --predict-1b` with all 6 ladder points (5m/40m/150m/350m/742m/1B) to sanity-check whether the token budget matched the FSS prediction.
3. For Phase 5 GSPO RL + process reward, resume from `final.pt` — that's the post-training foundation the RL stage expects.